# R05. Lazy imports

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/r05-lazy-imports/r05.ipynb)

R03 and R04 both assumed the same thing: when an import statement finishes, the module has been found, read and run. 3.15 adds one word that makes that untrue.

`lazy import json` binds the name and stops. The finding, the reading and the running happen later, the first time something reads the name back, and if nothing ever does then they never happen at all.

![a comparison of the four steps a plain import takes against the two steps a lazy import takes](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r05-lazy-imports/diagrams/two-spellings.svg)

## About the source references

Now and then this lesson points at CPython's own source, like this: `Python/import.c:3883-3895@v3.15.0rc1`.

Read it as three parts: the file, the lines, and the release those line numbers belong to. Sometimes there is a fourth part after a `#`, which is the name of the thing those lines are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Lazy imports are new in 3.15, so on 3.14, or in a browser tab, the cells that need the keyword say so and skip themselves. Everything else runs anywhere. Nothing here starts a second process or a second thread, so the threading question at the end is answered by recordings taken in containers instead.

## Which interpreter is this

In [ ]:
import pyxray

pyxray.show()

## What an unused import costs

Start with the problem rather than the feature. An import line at the top of a file is a promise that the module will be needed. Quite often it is not, and it gets paid for anyway.

Importing xml.etree.ElementTree brings in most of a package and a C extension underneath it, and a program that never parses any XML carries every one of those modules for its whole run

In [ ]:
import importlib
import time

FAMILY = ("xml", "_elementtree", "pyexpat")

before = set(sys.modules)
started = time.perf_counter()
importlib.import_module("xml.etree.ElementTree")
taken = (time.perf_counter() - started) * 1000
arrived = set(sys.modules) - before
family = sorted(name for name in sys.modules if name.startswith(FAMILY))

print(f"  what that one import line cost here: {taken:.1f} ms")
print(f"  modules it needs, all told: {len(family)}")
print("  which are:", ", ".join(family))
print(f"  how many of those this cell had to load: {len(arrived)}")
print()
print(f"  modules this process is now carrying: {len(sys.modules)}")

> **Version note.** the milliseconds are a measurement on your machine, the fourth line is zero if the runtime had already imported all of this for its own reasons, which a notebook kernel usually has, and 3.14 lists eight of these modules rather than nine because it has no xml.utils

That is one line in one file. A tool with twenty subcommands imports the machinery for all twenty every time you run one. The usual fix is to move the import into the function that needs it, which works and costs you the list of what a file depends on.

## What you had to do before

3.15 is not the first attempt. `importlib.util.LazyLoader` has been there since 3.5. It wraps a loader, hands back a module with a stand in class, and runs the real body the first time somebody touches it. That works, but it is five lines per module and more fragile than it looks.

A module made by LazyLoader replaces its own __getattribute__, so any attribute access at all wakes it up, including reading __name__ or __dict__ to see whether it is awake yet

In [ ]:
from importlib.util import LazyLoader, find_spec, module_from_spec


def the_old_way(name):
    """Defer a module the way you had to before 3.15, with a hand made spec and five lines."""
    spec = find_spec(name)
    spec.loader = LazyLoader(spec.loader)
    module = module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def names_in(module):
    """Count what a module holds without going through the machinery that would wake it up."""
    return len(object.__getattribute__(module, "__dict__"))


try:
    wave = the_old_way("wave")
    print("  what a lazy loader hands you:", type(wave).__name__)
    print("  in sys.modules already:      ", "wave" in sys.modules)
    print("  names it holds so far:       ", names_in(wave))
    print()
    print("  read one attribute of it, and not an interesting one:", wave.__name__)
    print("  the class it has now:        ", type(wave).__name__)
    print("  names it holds now:          ", names_in(wave))
except Exception as why:
    print("  this runtime cannot build a lazy loader here:", type(why).__name__, why)

> **Version note.** the two counts are the number of names the wave module holds before and after it runs, which moves whenever the module itself changes between releases, and a runtime that cannot find a spec for wave reports that instead

Reading `wave.__name__` was enough. `_LazyModule` overrides `__getattribute__` rather than `__getattr__` [Lib/importlib/util.py:167-205@v3.15.0rc1#_LazyModule](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/importlib/util.py#L167-L205), so there is no harmless look, which is why the cell reaches for `object.__getattribute__` to count names without disturbing anything. The other cost is `sys.modules[name] = module` up front: from then on every other importer gets your half built object. The new mechanism has neither problem, because it is not built out of modules at all.

## One word, and two bits in the opcode

`lazy` is a [soft keyword](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#soft-keyword), only a keyword directly in front of `import` or `from`, so every variable called `lazy` still works. The grammar arranges that with one lookahead [Grammar/python.gram:121-124@v3.15.0rc1#simple_stmt](https://github.com/python/cpython/blob/v3.15.0rc1/Grammar/python.gram#L121-L124) and an optional capture in the rule [Grammar/python.gram:227-236@v3.15.0rc1#import_name](https://github.com/python/cpython/blob/v3.15.0rc1/Grammar/python.gram#L227-L236).

What the compiler does is smaller than you would guess. There is no new opcode. `IMPORT_NAME` already took a name index as its argument, so 3.15 shifts that up by two bits and uses the two underneath to say which kind of import this is [Python/codegen.c:2902-2933@v3.15.0rc1#codegen_import](https://github.com/python/cpython/blob/v3.15.0rc1/Python/codegen.c#L2902-L2933).

A lazy import compiles to the same IMPORT_NAME opcode as a plain one, with the low two bits of the argument set to 1 for lazy and 2 for forced eager, and dis prints the difference

In [ ]:
import dis
import types

IN_A_FUNCTION = """
def f():
    import json
"""

IN_A_TRY = """
try:
    import json
except ImportError:
    pass
"""

FOUR = [
    ("plain, at module scope", "import json"),
    ("with the keyword", "lazy import json"),
    ("plain, inside a function", IN_A_FUNCTION),
    ("plain, inside a try block", IN_A_TRY),
]


def import_steps(code):
    """Every IMPORT_NAME in this code object and in the code objects nested inside it."""
    found = [step for step in dis.get_instructions(code) if step.opname == "IMPORT_NAME"]
    for const in code.co_consts:
        if isinstance(const, types.CodeType):
            found.extend(import_steps(const))
    return found


if not hasattr(sys, "lazy_modules"):
    print("  lazy is a word 3.15 added, so the second of these four does not compile here")
else:
    for label, source in FOUR:
        for step in import_steps(compile(source, "<demo>", "exec")):
            bits = step.arg & 3
            print(f"  {label:26} oparg {step.arg:3}  bits {bits:02b}  dis says {step.argrepr!r}")

> **Version note.** 3.14 has no lazy keyword, so the whole cell prints one line saying so and none of the four spellings get compiled there at all

![a table of the three values the low two bits of the IMPORT_NAME argument can take and what dis prints for each](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r05-lazy-imports/diagrams/the-two-bits.svg)

Bits `00` is the ordinary import and `01` is the keyword. Bits `10` is the compiler saying this one must happen now whatever anything else thinks, and it picks that for any import inside a function, a class body or a `try` block. The eval loop splits the argument the same way [Python/bytecodes.c:3497-3517@v3.15.0rc1#IMPORT_NAME](https://github.com/python/cpython/blob/v3.15.0rc1/Python/bytecodes.c#L3497-L3517).

## A workshop of our own

Everything below needs modules nothing else here has imported, because a name already in `sys.modules` never gets deferred, so the next cell writes throwaway modules into a temporary directory on `sys.path`.

It also gets around the version problem. `lazy import` is a syntax error on 3.14, so this notebook cannot contain one literally. The sources are compiled at run time and run with a single namespace dict, which is what the interpreter checks for when it asks whether a frame is at module scope [Python/ceval.c:3059-3064@v3.15.0rc1#is_lazy_import_module_level](https://github.com/python/cpython/blob/v3.15.0rc1/Python/ceval.c#L3059-L3064).

In [ ]:
import pathlib
import tempfile

LAZY = hasattr(sys, "lazy_modules")
WORKSHOP = pathlib.Path(tempfile.mkdtemp())
sys.path.insert(0, str(WORKSHOP))


def make_module(name, body):
    """Write a module of our own, so nothing else in this process has already imported it."""
    (WORKSHOP / f"{name}.py").write_text(body)


def module_level(source):
    """Run source the way a module body runs, and hand back the namespace it filled in."""
    namespace = {"__name__": "demo"}
    exec(compile(source, "<demo>", "exec"), namespace)
    return namespace


print("  a scratch directory to write modules into:", WORKSHOP.is_dir())
print("  this interpreter has lazy imports:        ", LAZY)

> **Version note.** the second line reads False on 3.14, and every cell below it takes its short branch

## What the statement leaves behind

Now the question the whole feature turns on. If the module body has not run, what is sitting under the name?

Not a module, and not a half built one either. It is a small object of its own kind, five fields long: the builtins it was declared in, the name, the attribute if this was a `from` import, and the code object and instruction offset of the line that declared it [Include/internal/pycore_lazyimportobject.h:17-25@v3.15.0rc1#PyLazyImportObject](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_lazyimportobject.h#L17-L25). The last two matter later.

A lazy import binds an object of type lazy_import, leaves sys.modules untouched, adds the name to sys.lazy_modules, and offers exactly one public method

In [ ]:
PAPERS = """
print("      (the papers module body is running now)")
STAMP = "signed"
"""

make_module("papers", PAPERS)

if LAZY:
    space = module_level("lazy import papers")
    print("  the statement has finished, and nothing printed above this line")
    print("  the name it bound is a:    ", type(space["papers"]).__name__)
    print("  its repr:                  ", repr(space["papers"]))
    print("  papers in sys.modules:     ", "papers" in sys.modules)
    print("  papers in sys.lazy_modules:", "papers" in sys.lazy_modules)
    offers = [one for one in dir(space["papers"]) if not one.startswith("_")]
    print("  everything the placeholder offers:", offers)
else:
    print("  there is nothing to bind here, because the keyword is a 3.15 thing")

> **Version note.** on 3.14 this prints the one line from its else branch and binds nothing

That is the whole object. It is not in `sys.modules`, so nobody else can trip over it, and unlike `_LazyModule` you can look at it as much as you like without setting it off. `sys.lazy_modules` is the other half of the bookkeeping, a set of every name still waiting. The one public method, `resolve`, forces the import and hands the module back without touching the namespace it came out of.

## What wakes it up

An [import placeholder](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#import-placeholder) is only useful if something turns it into a module at the right moment, and only two opcodes know about placeholders at all. Reading a bare name is one, and `LOAD_NAME` writes the resolved module back into globals so the next read is an ordinary lookup [Python/bytecodes.c:2280-2298@v3.15.0rc1#LOAD_NAME](https://github.com/python/cpython/blob/v3.15.0rc1/Python/bytecodes.c#L2280-L2298). Reading an attribute of a module is the other [Objects/moduleobject.c:1330-1360@v3.15.0rc1#_PyImport_LoadLazyImportTstate](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/moduleobject.c#L1330-L1360). Anything that goes around those two sees the placeholder.

Dict lookups, membership tests and reprs all leave the placeholder alone, while reading the name resolves it, and a placeholder copied into another variable resolves when that variable is read rather than when it was copied

In [ ]:
LEDGER = """
print("      (the ledger module body is running now)")
TOTAL = 7
"""

USE_PAPERS = """
lazy import papers
kept = papers.STAMP
"""

make_module("ledger", LEDGER)

if LAZY:
    print("  looking the name up in the namespace:", type(space["papers"]).__name__)
    print("  asking whether it is in there:       ", "papers" in space)
    print("  printing its repr:                   ", repr(space["papers"]))
    print("  after all three it is still a:       ", type(space["papers"]).__name__)
    print()
    print("  now read the name the way a module body would:")
    used = module_level(USE_PAPERS)
    print("  what came back:           ", used["kept"])
    print("  papers in sys.modules now:", "papers" in sys.modules)
    print()
    books = module_level("lazy import ledger")
    escaped = books["ledger"]
    print("  that assignment was a dict lookup, so ledger has run:", "ledger" in sys.modules)
    print("  now read the name we assigned to:", type(escaped).__name__)
    print("  ledger has run now:              ", "ledger" in sys.modules)
    print("  the entry it was copied out of:  ", type(books["ledger"]).__name__)
else:
    print("  no placeholder here to poke at")

> **Version note.** on 3.14 the else branch runs and neither of the two workshop modules is ever imported

![a table of five things you can do with a lazy name and whether each of them wakes the placeholder up](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r05-lazy-imports/diagrams/what-wakes-it.svg)

The last two lines are worth staring at. `escaped` came out by a dict lookup, which did nothing, and reading `escaped` resolved it. But the entry it was copied out of is still a placeholder, because nothing knew where the copy came from.

One more thing happens under the covers, and it is the one part of this lesson you cannot see from Python. The specialising interpreter refuses to specialise a global or a module attribute whose value is a placeholder [Python/specialize.c:1364-1374@v3.15.0rc1#SPEC_FAIL_ATTR_MODULE_LAZY_VALUE](https://github.com/python/cpython/blob/v3.15.0rc1/Python/specialize.c#L1364-L1374).

An unresolved placeholder blocks LOAD_GLOBAL_MODULE and LOAD_ATTR_MODULE from specialising

## Where the rules live

You can only write `lazy import` at module scope, and not inside a `try` block. The refusal is not where you would look first. It comes from the [symbol table](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#symbol-table), not the code generator [Python/symtable.c:1846-1876@v3.15.0rc1#check_lazy_import_context](https://github.com/python/cpython/blob/v3.15.0rc1/Python/symtable.c#L1846-L1876), because that pass already knows whether it is inside a function, a class or a `try`. Star imports are refused a few hundred lines further down [Python/symtable.c:2185-2196@v3.15.0rc1#ImportFrom](https://github.com/python/cpython/blob/v3.15.0rc1/Python/symtable.c#L2185-L2196), the code generator keeps its own copy of the check as a backstop [Python/codegen.c:2891-2900@v3.15.0rc1#codegen_validate_lazy_import](https://github.com/python/cpython/blob/v3.15.0rc1/Python/codegen.c#L2891-L2900), and writing the words the wrong way round gets its own grammar rule [Grammar/python.gram:1446-1449@v3.15.0rc1#invalid_import_from](https://github.com/python/cpython/blob/v3.15.0rc1/Grammar/python.gram#L1446-L1449).

Each of the four ways of misusing the keyword produces a different message naming the specific thing that is wrong, rather than one generic syntax error

In [ ]:
LAZY_IN_A_FUNCTION = """
def f():
    lazy import json
"""

LAZY_IN_A_CLASS = """
class C:
    lazy import json
"""

LAZY_IN_A_TRY = """
try:
    lazy import json
except ImportError:
    pass
"""

ATTEMPTS = [
    ("inside a function", LAZY_IN_A_FUNCTION),
    ("inside a class body", LAZY_IN_A_CLASS),
    ("inside a try block", LAZY_IN_A_TRY),
    ("asking for everything", "lazy from json import *"),
    ("the words the wrong way round", "from json lazy import dumps"),
    ("at module scope, which is fine", "lazy import json"),
]

for label, source in ATTEMPTS:
    try:
        compile(source, "<demo>", "exec")
    except SyntaxError as complaint:
        print(f"  {label:31} {complaint.msg}")
    else:
        print(f"  {label:31} compiles")

> **Version note.** 3.14 does not know the word at all, so all six lines there read invalid syntax, including the last one, which is the only one that is meant to work

The `try` block rule is the one to remember. A deferred import can fail at any point later, so an `except ImportError` around the statement would be a promise the interpreter cannot keep. Functions and classes are refused because placeholders live in a globals dict and a function's locals are not one, so there would be nowhere to put it. The runtime checks that a second time [Python/import.c:4538-4545@v3.15.0rc1#f_globals](https://github.com/python/cpython/blob/v3.15.0rc1/Python/import.c#L4538-L4545).

## Turning it on without the keyword

Rewriting every import in a codebase is not much of a migration path, so there are two other ways in. The first is a list: give a module a `__lazy_modules__` at the top, and plain imports of the names on it are deferred [Python/ceval.c:3021-3057@v3.15.0rc1#check_lazy_import_compatibility](https://github.com/python/cpython/blob/v3.15.0rc1/Python/ceval.c#L3021-L3057). The second is a filter, a callable of your own that the runtime asks about every candidate [Python/import.c:4547-4571@v3.15.0rc1#filter](https://github.com/python/cpython/blob/v3.15.0rc1/Python/import.c#L4547-L4571). It gets the importing module, the absolute name and the fromlist, and a false answer means an ordinary import.

Behind both is a mode, set with `-X lazy_imports=all` [Python/initconfig.c:2456-2487@v3.15.0rc1#config_init_lazy_imports](https://github.com/python/cpython/blob/v3.15.0rc1/Python/initconfig.c#L2456-L2487) and readable at run time [Python/sysmodule.c:2841-2855@v3.15.0rc1#set_lazy_imports](https://github.com/python/cpython/blob/v3.15.0rc1/Python/sysmodule.c#L2841-L2855). A stock interpreter starts in `normal`, where the keyword and the list work and nothing else changes.

![three boxes in a row naming the three source files that each get a say in whether an import is deferred](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r05-lazy-imports/diagrams/who-decides.svg)

A filter installed with sys.set_lazy_imports_filter is asked about every lazy import with three arguments, and returning false for one name makes that one import eagerly while the other stays deferred

In [ ]:
TWO_NAMES = """
lazy import cutlery
lazy import napkins
"""

BY_LIST = """
__lazy_modules__ = ["candles"]
import candles
import binascii
"""

make_module("cutlery", "SPOONS = 4")
make_module("napkins", "COUNT = 12")
make_module("candles", "LIT = True")

if LAZY:
    print("  the mode this interpreter started in:", sys.get_lazy_imports())
    asked = []

    def only_cutlery(importer, imported, fromlist):
        """Keep the deferral for one name and hand the other one back to the ordinary loader."""
        asked.append((importer, imported, fromlist))
        return imported == "cutlery"

    sys.set_lazy_imports_filter(only_cutlery)
    try:
        table = module_level(TWO_NAMES)
    finally:
        sys.set_lazy_imports_filter(None)

    for call in asked:
        print("  the filter was asked about:", call)
    print("  cutlery came out as:", type(table["cutlery"]).__name__)
    print("  napkins came out as:", type(table["napkins"]).__name__)
    print()
    older = module_level(BY_LIST)
    print("  with __lazy_modules__ and no keyword, candles is:", type(older["candles"]).__name__)
    print("  and binascii, which is not on that list, is:     ", type(older["binascii"]).__name__)
else:
    print("  no modes and no filter on this interpreter")

> **Version note.** 3.14 has no sys.get_lazy_imports and no filter to install, so the else branch runs there and the three workshop modules are written out and left alone

## When the module raises

Deferring the work moves the failure with it. If the module body raises, it raises wherever the name was read, which might be a long way from the import line.

CPython gives you both places. It builds a second exception, an `ImportError` naming the lazy import, reconstructs a traceback frame from the code object and offset the placeholder has carried since it was created, and hangs the whole thing on the real exception as its cause [Python/import.c:4055-4086@v3.15.0rc1#PyException_SetCause](https://github.com/python/cpython/blob/v3.15.0rc1/Python/import.c#L4055-L4086).

A failure inside a deferred module arrives as the exception the module raised, with an ImportError attached as its __cause__ whose traceback points at the lazy import line rather than at the use site

In [ ]:
import traceback

BAD = """
lazy import takings
kept = takings.TOTAL
"""

make_module("takings", "TOTAL = 1 / 0")


def where(error):
    """The file and line of the last frame in an exception's traceback."""
    last = traceback.extract_tb(error.__traceback__)[-1]
    return f"{pathlib.Path(last.filename).name} line {last.lineno}"


if LAZY:
    try:
        module_level(BAD)
    except ZeroDivisionError as went_wrong:
        cause = went_wrong.__cause__
        print("  what reached us:       ", type(went_wrong).__name__, went_wrong)
        print("  raised at:             ", where(went_wrong))
        print("  what it names as cause:", type(cause).__name__)
        print("                         ", cause)
        print("  and that points at:    ", where(cause))
else:
    print("  no two part traceback here, because there is nothing deferred to fail")

> **Version note.** on 3.14 the takings module is written out and never imported, so nothing fails

Two files and two line numbers, which between them tell the whole story: the division happened in `takings.py`, and the reason anybody ran it then is the `lazy import` at the top of the other file.

The resolution path guards against one more thing. If waking a placeholder leads back to waking the same one you get an `ImportCycleError` rather than a hang, because the set of names being resolved is kept on the interpreter and checked on the way in [Python/import.c:3897-3925@v3.15.0rc1#lazy_importing_modules](https://github.com/python/cpython/blob/v3.15.0rc1/Python/import.c#L3897-L3925).

## What the standard library already defers

This is not a feature waiting for users. 3.15 ships with it on in about thirty files, including `collections`, `typing`, `inspect`, `dataclasses`, `argparse` and `contextlib`. Four are in `site.py`, and they are why a stock interpreter has anything in `sys.lazy_modules` before you write a line [Lib/site.py:46-54@v3.15.0rc1#lazy](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/site.py#L46-L54). `collections` defers two functions it needs for one method each [Lib/collections/__init__.py:35-36@v3.15.0rc1#nlargest](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/collections/__init__.py#L35-L36).

That has a knock on effect. Resolving one deferred import runs a module body, and that body can declare deferred imports of its own, so the set grows while you drain it.

Resolving one placeholder runs a module body that can declare placeholders of its own, so a name can appear in sys.lazy_modules as a result of resolving a different one

In [ ]:
OUTER = """
lazy import inner
DEPTH = 1
"""

USE_OUTER = """
lazy import outer
kept = outer.DEPTH
"""

make_module("inner", "DEPTH = 2")
make_module("outer", OUTER)

if LAZY:
    waiting = sorted(sys.lazy_modules)
    print("  names this process has waiting right now:", len(waiting))
    print("  four of them:", ", ".join(waiting[:4]))
    print()
    module_level("lazy import outer")
    print("  inner is waiting before we touch outer:", "inner" in sys.lazy_modules)
    used = module_level(USE_OUTER)
    print("  outer has now run and gave us:         ", used["kept"])
    print("  and inner is waiting now:              ", "inner" in sys.lazy_modules)
    print("  while inner itself is in sys.modules:  ", "inner" in sys.modules)
else:
    print("  sys.lazy_modules does not exist on this interpreter")

> **Version note.** the count of waiting names and the four shown depend entirely on what the runtime has imported for its own reasons before the cell runs, so a notebook kernel reports a longer list than a bare interpreter does, and the four lines underneath are the same everywhere

The last two lines are the point. `inner` is waiting, which means `outer` ran, and `inner` is not in `sys.modules`, which means it did not. Deferral survives one level down.

## What it saves

The same measurement twice, once in this process and once as a real startup in a container.

A file with twelve imports at the top that uses one runs several times faster with them deferred, and eleven of the twelve module bodies never run

In [ ]:
BODY = "total = 0\nfor i in range(200000):\n    total += i\nTOTAL = total\n"

if LAZY:
    heavy = [f"weight_{index}" for index in range(12)]
    light = [f"feather_{index}" for index in range(12)]
    for name in heavy + light:
        make_module(name, BODY)

    plain = "\n".join(f"import {name}" for name in heavy) + "\nkept = weight_3.TOTAL\n"
    deferred = "\n".join(f"lazy import {name}" for name in light) + "\nkept = feather_3.TOTAL\n"

    started = time.perf_counter()
    first = module_level(plain)
    plain_ms = (time.perf_counter() - started) * 1000

    started = time.perf_counter()
    second = module_level(deferred)
    lazy_ms = (time.perf_counter() - started) * 1000

    print(f"  twelve imports and one use, as written: {plain_ms:8.2f} ms")
    print(f"  the same twelve, deferred:              {lazy_ms:8.2f} ms")
    print("  both files ended up with the same answer:", first["kept"] == second["kept"])
    print("  modules of the first set that ran:      ", sum(n in sys.modules for n in heavy))
    print("  modules of the second set that ran:     ", sum(n in sys.modules for n in light))
else:
    print("  nothing to defer here, so there is nothing to time")

> **Version note.** the two timings are measurements on your machine and the ratio between them depends on how slow your filesystem is, since most of what the first number pays for is opening and compiling twelve files that nothing needed

Twelve is small for a real program. Here is the same shape as a whole process startup.

What does a program get back for not importing what it turns out not to need?

```python
"""What deferring an import until somebody touches the name is worth.

A program that imports twelve standard library modules at the top and then uses one of them is
not a strawman, it is what most command line tools look like. PEP 810 adds a spelling that says
bind this name now and do the work later, and a mode that applies the same rule to every plain
import in the file, so the cost of the eleven you did not need can be measured rather than
argued about.

The child processes here run the same source twice, once with the default mode and once with
-X lazy_imports=all. Three kinds of number come back. How many modules end up in sys.modules is
a count, so it is the same on any machine. How many code objects the interpreter reads off disk
is also a count, and -v prints one line for each. Wall clock is the one that depends on the
machine, and it is measured with the two cases alternating, because running one case forty
times and then the other measures the page cache instead.
"""

import statistics
import subprocess
import sys
import time

ROUNDS = 40
LAZY = ["-X", "lazy_imports=all"]
MODULES = (
    "argparse",
    "csv",
    "dataclasses",
    "email.parser",
    "http.client",
    "json",
    "logging",
    "sqlite3",
    "typing",
    "unittest",
    "urllib.request",
    "xml.etree.ElementTree",
)
BODY = "\n".join("import " + name for name in MODULES)
WORK = "print(json.dumps({'used': 1}))"
COUNT = "import sys; print(len(sys.modules), len(sys.lazy_modules))"


def child(flags, args):
    """Run this same interpreter again with those flags and hand back the finished process."""
    return subprocess.run(
        [sys.executable, *flags, *args], capture_output=True, text=True, check=True
    )


def asked(question):
    """Ask a fresh interpreter one question and hand back the single line it prints."""
    return child([], ["-c", "import sys; print({})".format(question)]).stdout.strip()


def loaded(flags):
    """Import the twelve, use one, and ask how many modules the process ended up with."""
    printed = child(flags, ["-c", BODY + "\n" + WORK + "\n" + COUNT + "\n"]).stdout
    return [int(part) for part in printed.splitlines()[-1].split()]


def files_read(flags):
    """Count the code objects the run reads off disk, which -v prints one line for."""
    printed = child(flags, ["-v", "-c", BODY + "\n" + WORK + "\n"]).stderr
    return sum(1 for line in printed.splitlines() if line.startswith("# code object from"))


def run_ms(flags):
    """Wall clock milliseconds for one whole run, startup included."""
    started = time.perf_counter()
    child(flags, ["-c", BODY + "\n" + WORK + "\n"])
    return (time.perf_counter() - started) * 1000


def alternating(rounds):
    """Measure both cases once each per round, so neither one gets the cold cache every time."""
    got = {"eager": [], "lazy": []}
    for _ in range(rounds):
        got["eager"].append(run_ms([]))
        got["lazy"].append(run_ms(LAZY))
    return got


print("modules this program imports at the top:", len(MODULES))
print("modules it actually uses:", 1)
print("the mode this build starts in:", asked("sys.get_lazy_imports()"))
print("names the standard library defers at startup:", asked("len(sys.lazy_modules)"))
print("which names those are:", asked("sorted(sys.lazy_modules)"))

eager_total, eager_waiting = loaded([])
lazy_total, lazy_waiting = loaded(LAZY)
print("modules in sys.modules at the end, eager:", eager_total)
print("modules in sys.modules at the end, lazy:", lazy_total)
print("names still waiting in sys.lazy_modules, eager:", eager_waiting)
print("names still waiting in sys.lazy_modules, lazy:", lazy_waiting)
print("code objects read off disk, eager:", files_read([]))
print("code objects read off disk, lazy:", files_read(LAZY))

runs = alternating(ROUNDS)
fast_eager, fast_lazy = min(runs["eager"]), min(runs["lazy"])
print("~ fastest run with plain imports: {:.1f} ms".format(fast_eager))
print("~ fastest run with lazy imports: {:.1f} ms".format(fast_lazy))
print("~ middle run with plain imports: {:.1f} ms".format(statistics.median(runs["eager"])))
print("~ middle run with lazy imports: {:.1f} ms".format(statistics.median(runs["lazy"])))
share = (1 - fast_lazy / fast_eager) * 100
print("~ share of the run that deferring gives back: {:.1f} percent".format(share))
```

```text
modules this program imports at the top: 12
modules it actually uses: 1
the mode this build starts in: normal
names the standard library defers at startup: 4
which names those are: ['locale', 'pkgutil', 'traceback', 'warnings']
modules in sys.modules at the end, eager: 189
modules in sys.modules at the end, lazy: 54
names still waiting in sys.lazy_modules, eager: 16
names still waiting in sys.lazy_modules, lazy: 57
code objects read off disk, eager: 110
code objects read off disk, lazy: 15
~ fastest run with plain imports: 190.4 ms
~ fastest run with lazy imports: 48.9 ms
~ middle run with plain imports: 227.1 ms
~ middle run with lazy imports: 54.9 ms
~ share of the run that deferring gives back: 74.3 percent
```

That ran on Python 3.15.0rc1 in the release build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:release@sha256:fb55d6afcf053c974de6447fafbd2be6af20cdb9f596e25a0445607b8af981e3`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:release@sha256:fb55d6afcf053c974de6447fafbd2be6af20cdb9f596e25a0445607b8af981e3 python3 -` takes the program on standard input.

![two bars comparing the milliseconds a startup takes with twelve plain imports against the same file with them deferred](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r05-lazy-imports/diagrams/what-deferring-saves.svg)

Three quarters of the run went with the eleven imports nothing needed. The counts underneath are the honest part: fifty four modules in `sys.modules` rather than a hundred and eighty nine, and fifteen code objects read off disk rather than a hundred and ten.

## The lock nobody mentions

This part is not in the release notes, and it only shows up on a free threaded build.

An ordinary import takes a [module lock](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#module-lock), one per module name, so two threads importing two different modules do not wait for each other. Waking a placeholder does not use that lock. It takes the interpreter wide import lock and holds it for the whole resolution, module body included [Python/import.c:3883-3895@v3.15.0rc1#_PyImport_LoadLazyImportTstate](https://github.com/python/cpython/blob/v3.15.0rc1/Python/import.c#L3883-L3895), a recursive mutex on the interpreter [Python/import.c:150-158@v3.15.0rc1#_PyImport_AcquireLock](https://github.com/python/cpython/blob/v3.15.0rc1/Python/import.c#L150-L158). The comment says why: to serialise reification. Two threads waking two different placeholders take turns.

![a comparison of the per name lock an ordinary import takes against the interpreter wide lock a lazy import takes](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/r05-lazy-imports/diagrams/which-lock.svg)

On a normal build none of this shows, because the global interpreter lock serialises the module bodies anyway.

Do two threads waking up two different deferred imports wait for each other?

```python
"""What happens when several threads wake up deferred imports at once.

An ordinary import takes a lock on the name being imported, so two threads importing two
different modules do not wait for each other. Waking up a deferred import is a different code
path, and _PyImport_LoadLazyImportTstate takes the interpreter wide import lock instead, and
holds it for as long as the module body runs.

That difference is invisible on a build with the global interpreter lock, because nothing runs
in parallel there anyway. On a free threaded build it should be visible, and this program is
the measurement. Both cases are reported as cores kept busy, which is processor time divided by
wall clock, because that is the only honest way to compare several threads against one on a
machine whose scheduler moves work between cores.

Each generated module burns processor time in its body rather than sleeping, so a case that
really overlaps reads well above one core and a case that takes turns reads about one.
"""

import importlib
import os
import pathlib
import sys
import tempfile
import threading
import time

THREADS = 4
BURN = 3_000_000
SOURCE = "total = 0\nfor i in range({burn}):\n    total += i\nBURNED = total\n"

folder = pathlib.Path(tempfile.mkdtemp())
sys.path.insert(0, str(folder))


def write_modules(prefix):
    """Write one module per thread, each of which does real arithmetic in its body."""
    names = []
    for index in range(THREADS):
        name = "{}_{}".format(prefix, index)
        (folder / (name + ".py")).write_text(SOURCE.format(burn=BURN))
        names.append(name)
    return names


def toucher(name):
    """Build a function whose body reads that global, because reading it is what wakes it up."""
    namespace = {}
    body = "def touch():\n    return {}.BURNED\n".format(name)
    exec(compile(body, "<touch>", "exec"), globals(), namespace)
    return namespace["touch"]


def together(jobs):
    """Run one thread per job, all released at the same instant, and time the whole batch."""
    ready = threading.Barrier(len(jobs))
    seen = []

    def work(job):
        ready.wait()
        seen.append(job())

    threads = [threading.Thread(target=work, args=(job,)) for job in jobs]
    wall = time.perf_counter()
    cpu = time.process_time()
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    wall = time.perf_counter() - wall
    cpu = time.process_time() - cpu
    return len(seen), wall, cpu / wall


def one_thread():
    """The same arithmetic on one thread, as a reading of what one busy core looks like here."""
    wall = time.perf_counter()
    cpu = time.process_time()
    total = 0
    for i in range(BURN):
        total += i
    wall = time.perf_counter() - wall
    return (time.process_time() - cpu) / wall, wall


plain = write_modules("plain")
deferred = write_modules("deferred")
exec(compile("\n".join("lazy import " + name for name in deferred), "<declare>", "exec"), globals())

print("threads:", THREADS)
print("processors this container can see:", os.process_cpu_count())
print("gil enabled:", sys._is_gil_enabled())
print("names declared and still waiting:", sum(1 for n in deferred if n in sys.lazy_modules))
print("any of them in sys.modules yet:", any(n in sys.modules for n in deferred))

control_cores, control_wall = one_thread()

done, wall_plain, cores_plain = together([lambda n=n: importlib.import_module(n) for n in plain])
print("threads that finished a plain import:", done)

done, wall_lazy, cores_lazy = together([toucher(n) for n in deferred])
print("threads that woke up a deferred import:", done)
print("all of them in sys.modules now:", all(n in sys.modules for n in deferred))
print("names still waiting afterwards:", sum(1 for n in deferred if n in sys.lazy_modules))

print("~ cores kept busy by one thread doing the arithmetic: {:.2f}".format(control_cores))
print("~ cores kept busy importing four modules the plain way: {:.2f}".format(cores_plain))
print("~ cores kept busy waking four deferred imports: {:.2f}".format(cores_lazy))
print("~ wall clock for one thread, ms: {:.0f}".format(control_wall * 1000))
print("~ wall clock for the plain imports, ms: {:.0f}".format(wall_plain * 1000))
print("~ wall clock for the deferred ones, ms: {:.0f}".format(wall_lazy * 1000))
```

```text
threads: 4
processors this container can see: 4
gil enabled: True
names declared and still waiting: 4
any of them in sys.modules yet: False
threads that finished a plain import: 4
threads that woke up a deferred import: 4
all of them in sys.modules now: True
names still waiting afterwards: 0
~ cores kept busy by one thread doing the arithmetic: 0.99
~ cores kept busy importing four modules the plain way: 1.02
~ cores kept busy waking four deferred imports: 0.99
~ wall clock for one thread, ms: 198
~ wall clock for the plain imports, ms: 2137
~ wall clock for the deferred ones, ms: 2066
```

That ran on Python 3.15.0rc1 in the release build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:release@sha256:fb55d6afcf053c974de6447fafbd2be6af20cdb9f596e25a0445607b8af981e3`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:release@sha256:fb55d6afcf053c974de6447fafbd2be6af20cdb9f596e25a0445607b8af981e3 python3 -` takes the program on standard input.

With the global interpreter lock gone, does waking up a deferred import scale?

```python
"""What happens when several threads wake up deferred imports at once.

An ordinary import takes a lock on the name being imported, so two threads importing two
different modules do not wait for each other. Waking up a deferred import is a different code
path, and _PyImport_LoadLazyImportTstate takes the interpreter wide import lock instead, and
holds it for as long as the module body runs.

That difference is invisible on a build with the global interpreter lock, because nothing runs
in parallel there anyway. On a free threaded build it should be visible, and this program is
the measurement. Both cases are reported as cores kept busy, which is processor time divided by
wall clock, because that is the only honest way to compare several threads against one on a
machine whose scheduler moves work between cores.

Each generated module burns processor time in its body rather than sleeping, so a case that
really overlaps reads well above one core and a case that takes turns reads about one.
"""

import importlib
import os
import pathlib
import sys
import tempfile
import threading
import time

THREADS = 4
BURN = 3_000_000
SOURCE = "total = 0\nfor i in range({burn}):\n    total += i\nBURNED = total\n"

folder = pathlib.Path(tempfile.mkdtemp())
sys.path.insert(0, str(folder))


def write_modules(prefix):
    """Write one module per thread, each of which does real arithmetic in its body."""
    names = []
    for index in range(THREADS):
        name = "{}_{}".format(prefix, index)
        (folder / (name + ".py")).write_text(SOURCE.format(burn=BURN))
        names.append(name)
    return names


def toucher(name):
    """Build a function whose body reads that global, because reading it is what wakes it up."""
    namespace = {}
    body = "def touch():\n    return {}.BURNED\n".format(name)
    exec(compile(body, "<touch>", "exec"), globals(), namespace)
    return namespace["touch"]


def together(jobs):
    """Run one thread per job, all released at the same instant, and time the whole batch."""
    ready = threading.Barrier(len(jobs))
    seen = []

    def work(job):
        ready.wait()
        seen.append(job())

    threads = [threading.Thread(target=work, args=(job,)) for job in jobs]
    wall = time.perf_counter()
    cpu = time.process_time()
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    wall = time.perf_counter() - wall
    cpu = time.process_time() - cpu
    return len(seen), wall, cpu / wall


def one_thread():
    """The same arithmetic on one thread, as a reading of what one busy core looks like here."""
    wall = time.perf_counter()
    cpu = time.process_time()
    total = 0
    for i in range(BURN):
        total += i
    wall = time.perf_counter() - wall
    return (time.process_time() - cpu) / wall, wall


plain = write_modules("plain")
deferred = write_modules("deferred")
exec(compile("\n".join("lazy import " + name for name in deferred), "<declare>", "exec"), globals())

print("threads:", THREADS)
print("processors this container can see:", os.process_cpu_count())
print("gil enabled:", sys._is_gil_enabled())
print("names declared and still waiting:", sum(1 for n in deferred if n in sys.lazy_modules))
print("any of them in sys.modules yet:", any(n in sys.modules for n in deferred))

control_cores, control_wall = one_thread()

done, wall_plain, cores_plain = together([lambda n=n: importlib.import_module(n) for n in plain])
print("threads that finished a plain import:", done)

done, wall_lazy, cores_lazy = together([toucher(n) for n in deferred])
print("threads that woke up a deferred import:", done)
print("all of them in sys.modules now:", all(n in sys.modules for n in deferred))
print("names still waiting afterwards:", sum(1 for n in deferred if n in sys.lazy_modules))

print("~ cores kept busy by one thread doing the arithmetic: {:.2f}".format(control_cores))
print("~ cores kept busy importing four modules the plain way: {:.2f}".format(cores_plain))
print("~ cores kept busy waking four deferred imports: {:.2f}".format(cores_lazy))
print("~ wall clock for one thread, ms: {:.0f}".format(control_wall * 1000))
print("~ wall clock for the plain imports, ms: {:.0f}".format(wall_plain * 1000))
print("~ wall clock for the deferred ones, ms: {:.0f}".format(wall_lazy * 1000))
```

```text
threads: 4
processors this container can see: 4
gil enabled: False
names declared and still waiting: 4
any of them in sys.modules yet: False
threads that finished a plain import: 4
threads that woke up a deferred import: 4
all of them in sys.modules now: True
names still waiting afterwards: 0
~ cores kept busy by one thread doing the arithmetic: 0.98
~ cores kept busy importing four modules the plain way: 3.60
~ cores kept busy waking four deferred imports: 0.98
~ wall clock for one thread, ms: 207
~ wall clock for the plain imports, ms: 1142
~ wall clock for the deferred ones, ms: 2602
```

That ran on Python 3.15.0rc1 in the freethreaded build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144 python3 -` takes the program on standard input.

Read the middle two numbers of each. On the ordinary build both cases keep about one core busy. On the free threaded build the plain imports keep 3.60 cores busy and the deferred ones keep 0.98, the same single core the one thread control measured.

On a free threaded build, four threads importing four different modules run about three and a half times over, while four threads waking four different deferred imports run one at a time, because reification takes the interpreter wide import lock

Worth knowing before turning `lazy_imports=all` on in a threaded program. Deferring moves import work off the startup path, but on a free threaded build it moves that work onto a lock the ordinary path does not use.

## Try it yourself

Four things, in rough order of how much you will learn.

Run `python -X lazy_imports=all -c "import sys; print(len(sys.modules), len(sys.lazy_modules))"` and compare it with the same command without the flag. That is the whole feature in one line.

Take the escaped placeholder cell and put the copy somewhere a bit further away, such as a list or an attribute of an object, and check what it takes to wake it up. The rule is only about the two opcodes, so a placeholder inside a list stays a placeholder no matter how many times you index it.

Write a filter that prints every candidate rather than deciding anything, install it, and import something big. It is the cheapest way to see what a real program actually asks for.

Read `Python/import.c:3897-3925` and work out what happens if a deferred module's body reads the very name that is being resolved. Then write it and check.

## What you now know

`lazy import json` binds a name and does nothing else. The finding, reading and running happen the first time something reads that name back, and never if nothing does.

It compiles to the same `IMPORT_NAME` opcode as a plain import, with two spare bits in the argument saying lazy, eager or ordinary. There is no new opcode.

What gets bound is not a module. It is a five field placeholder that is not in `sys.modules`, that you can look at without setting it off, and whose name is in `sys.lazy_modules` until it resolves.

Only two opcodes resolve one: reading it as a bare name and reading it as an attribute of a module. Dict lookups, membership tests and reprs all leave it alone, and a placeholder copied into another variable resolves when that variable is read.

The scope rules are enforced by the symbol table, which is why `try` blocks are refused with a specific message rather than a generic syntax error.

Two other ways in exist for code you do not want to edit: a `__lazy_modules__` list at the top of a module, and a filter installed with `sys.set_lazy_imports_filter`.

A failure inside a deferred module arrives with a second exception attached as its cause, pointing at the `lazy import` line, reconstructed from information the placeholder was carrying all along.

It is worth about three quarters of a startup on a file that imports twelve modules and uses one, and about a hundred and thirty fewer modules loaded.

Resolution takes the interpreter wide import lock rather than a per name one, so on a free threaded build deferred imports do not scale across threads and ordinary ones do.

## What is next

R06 turns from the Python side of the runtime to the C side. Everything in these five lessons has an equivalent in the C API, and that API has tiers: a limited subset with a promise attached, a general one with no promise, and an internal one that is not for you at all.

Knowing which tier a function is in decides whether an extension you build against 3.15 still loads on 3.16, which is the same kind of question this lesson asked about when a module body runs, moved from time to versions.